# What’s 4 Dinner? v1.2.0 — Open-Source Agent Pipeline

This Google Colab notebook runs an autonomous nutritional research agent using **entirely open-source tools** on Google Colab's free T4 GPU. **No API keys, accounts, or credit cards are required.**

### Pipeline Components:
1. **DuckDuckGo Search (`duckduckgo-search`)**: Scrapes live local restaurant menus with zero API keys.
2. **Open-Source LLM (`Qwen/Qwen2.5-7B-Instruct`)**: Runs locally in 4-bit precision on Colab's free T4 GPU.
3. **Clinical Dietetic Audit**: Verifies GLP-1 gastric tolerance, low sodium (<500mg), and low calories (<450 kcal).
4. **JSON Output**: Compiles `meals.json` ready for direct import into the What's 4 Dinner PWA.

### Step 1: Check GPU & Install Lightweight Dependencies
> **Note:** Colab's pre-installed `torch` is left intact to avoid CUDA version conflicts.

In [ ]:
# Verify GPU hardware is active
!nvidia-smi

# Install only inference helper libraries (leaves pre-installed torch untouched)
!pip install -q -U transformers accelerate bitsandbytes duckduckgo-search

### Step 2: Live Local Web Search (Zero API Key Needed)

In [ ]:
from duckduckgo_search import DDGS
import json

LOCATION = "Summerville, SC"
SEARCH_QUERY = f"healthy dinner restaurant menu {LOCATION} grilled lean protein seafood"

print(f"[*] Searching DuckDuckGo for menus in {LOCATION}...")
search_context = ""
with DDGS() as ddgs:
    results = list(ddgs.text(SEARCH_QUERY, max_results=6))
    for i, item in enumerate(results, 1):
        snippet = f"Source [{i}]: {item.get('title', '')} - {item.get('body', '')}\n"
        search_context += snippet

print("[*] Retrieved Context:\n")
print(search_context[:650] + "...\n")

### Step 3: Load Quantized 7B Open-Source LLM onto GPU

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_id = "Qwen/Qwen2.5-7B-Instruct"

# 4-bit NF4 configuration fits comfortably in Colab's ~15 GB T4 VRAM (~5.5 GB used)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

print(f"[*] Downloading and loading {model_id} on GPU...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto"
)
print("[*] Model loaded and ready!")

### Step 4: Run Clinical Audit & Export `meals.json`

In [ ]:
system_prompt = """You are an expert clinical dietitian.
Audit the web search results and extract or formulate dinner options strictly matching:
1. GLP-1 therapy suitable: Lean protein, low gastric distress, no heavy creams or deep-fried oils.
2. Low Sodium: < 500mg sodium per entree.
3. Low Calorie: < 450 kcal total.
4. High Protein: >= 30g protein.

Respond ONLY with a valid JSON array matching this schema:
[
  {
    "id": "1",
    "type": "dine-out",
    "venue": "Restaurant Name",
    "location": "City / Neighborhood",
    "title": "Entree Name",
    "calories": 390,
    "protein": 38,
    "sodium": 340,
    "glp1": true,
    "lowSodium": true,
    "lowCal": true,
    "highProtein": true,
    "orderTip": "Specific server modification instruction",
    "address": "City, State or Street Address for Google Maps"
  }
]
Do NOT include any markdown or text outside the JSON array."""

user_prompt = f"Target Location: {LOCATION}\nWeb Context:\n{search_context}\n\nGenerate 5 compliant dinner options based on this area."

messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer([prompt_text], return_tensors="pt").to("cuda")

print("[*] Reasoning through clinical diet criteria...")
with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=1200,
        temperature=0.2,
        do_sample=False
    )

gen_tokens = [out[len(inp):] for inp, out in zip(inputs.input_ids, output_tokens)]
response = tokenizer.batch_decode(gen_tokens, skip_special_tokens=True)[0].strip()

if response.startswith("```json"): response = response[7:]
if response.startswith("```"): response = response[3:]
if response.endswith("```"): response = response[:-3]
response = response.strip()

try:
    parsed_data = json.loads(response)
    with open("meals.json", "w", encoding="utf-8") as f:
        json.dump(parsed_data, f, indent=2)
    print("\n[+] Generated valid meals.json:")
    print(json.dumps(parsed_data, indent=2))
except Exception as e:
    print("[!] Parsing note:", e)
    print("Raw Model Response:", response)

### Step 5: Download `meals.json`
Save the file locally so you can use the **Import JSON** button in the What's 4 Dinner PWA.

In [ ]:
from google.colab import files
files.download('meals.json')